In [ ]:
# fixar seed, instalar libs se necessário
RANDOM_STATE = 42

# bibliotecas principais
import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# try to import xgboost; install if missing
try:
    import xgboost as xgb
except Exception:
    !pip install xgboost -q
    import xgboost as xgb


In [ ]:
DATA_URL = 'https://github.com/thiagolmetne/Mvp---Machine-Learning-Analytics/blob/main/gpd_medalha-000000000000.csv'  # ex: 'https://raw.githubusercontent.com/usuario/repo/main/olimpiadas.csv'

if DATA_URL:
    df = pd.read_csv(DATA_URL)
else:
    # fallback: upload manual no Colab
    from google.colab import files
    print("Selecione o arquivo CSV local para upload (ou pressione Cancel para interromper).")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("Nenhum arquivo enviado. Defina DATA_URL ou faça upload do CSV.")
    # pega o primeiro arquivo enviado
    fname = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[fname]))

print("Shape:", df.shape)
df.head()


In [ ]:
# info e missing
display(df.info())
display(df.describe(include='all').T)
# colunas esperadas: ano, delegacao, nivel_renda, media_gdp, total_medalhas, qtd_paises
print("Missing por coluna:\n", df.isna().sum())

# target distribution
plt.figure(figsize=(8,4))
sns.histplot(df['total_medalhas'].fillna(0), bins=50, kde=False)
plt.title('Distribuição de total_medalhas')
plt.xlabel('total_medalhas')
plt.show()

# valores únicos de nivel_renda e delegacao contagem
print("nivel_renda únicas:", df['nivel_renda'].unique())
print("delegacao top 10:\n", df['delegacao'].value_counts().head(10))


In [ ]:
# converter colunas e limpeza inicial
df['ano'] = df['ano'].astype(int)
# garantir numericidade de media_gdp, total_medalhas, qtd_paises
for c in ['media_gdp','total_medalhas','qtd_paises']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# remover linhas sem delegacao ou sem ano
df = df.dropna(subset=['delegacao','ano']).reset_index(drop=True)
print("Após limpeza:", df.shape)


Markdown — Feature engineering breve """
Imputar nivel_renda faltante como 'Unknown'.
Transformar media_gdp com log1p por assimetria.
Codificar delegacao: manter top N delegações (por soma de medalhas) como one‑hot, agrupar resto como 'OUTROS'.
Usaremos qtd_paises e ano como features numéricas. """

Hiperparâmetros: RandomizedSearchCV para XGBoost
"""
Faremos ajuste moderado de hiperparâmetros no XGBoost (RandomizedSearchCV) para melhorar desempenho. Só rodar se desejar (pode demorar).
"""

In [ ]:
from scipy.stats import randint, uniform

xgb_pipe = models['xgb']
param_dist = {
    'model__n_estimators': randint(50,300),
    'model__max_depth': randint(3,10),
    'model__learning_rate': uniform(0.01,0.3),
    'model__subsample': uniform(0.6,0.4),
    'model__colsample_bytree': uniform(0.5,0.5)
}

search = RandomizedSearchCV(xgb_pipe, param_distributions=param_dist, n_iter=30, scoring='neg_mean_absolute_error',
                            cv=KFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE), n_jobs=-1, random_state=RANDOM_STATE, verbose=1)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)
best_xgb = search.best_estimator_

# avaliar no teste
best_res = evaluate_model(best_xgb, X_train, y_train, X_test, y_test, cv=4)
print("Tuned XGB test MAE:", best_res['test_mae'], "RMSE:", best_res['test_rmse'], "R2:", best_res['test_r2'])
results['xgb_tuned'] = best_res


Comparação e visualizações
"""
Comparar métricas e plotar predições vs valores reais para o melhor modelo.
"""

In [ ]:
# escolher melhor por menor test_mae
best_model_name = res_df.iloc[0]['model']
print("Best model:", best_model_name)
preds = results[best_model_name]['preds'] if 'preds' in results[best_model_name] else best_xgb.predict(X_test)
plt.figure(figsize=(6,6))
plt.scatter(y_test, preds, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Real total_medalhas')
plt.ylabel('Predito total_medalhas')
plt.title(f'Predito vs Real — {best_model_name}')
plt.show()

# mostrar tabela com alguns casos
out = test_df.copy()
out['pred'] = preds
display(out.sort_values('pred', ascending=False).head(10))


Ver feature importance do XGBoost

In [ ]:
# aplicar preprocessor fit para extrair feature names
preprocessor.fit(X_train)
# build feature names
num_names = num_features
ohe = preprocessor.named_transformers_['cat'].named_steps['ohe']
cat_names = ohe.get_feature_names_out(cat_features).tolist()
feature_names = num_names + cat_names

# se best_model for pipeline com xgb/rf, extrair o modelo
if 'xgb' in best_model_name:
    model_obj = (best_xgb.named_steps['model'] if 'best_xgb' in locals() else search.best_estimator_.named_steps['model'])
elif 'rf' in best_model_name:
    model_obj = models['rf'].fit(X_train, y_train).named_steps['model']
else:
    model_obj = None

if model_obj is not None:
    try:
        importances = model_obj.feature_importances_
        fi = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(20)
        plt.figure(figsize=(8,4))
        sns.barplot(x=fi.values, y=fi.index)
        plt.title('Feature importance (approx.)')
        plt.show()
    except Exception as e:
        print("Não foi possível extrair feature importance:", e)


Salvar modelo e conclusão
"""
Salve o melhor pipeline e escreva uma conclusão resumida (o que foi feito, métricas, limitações, próximos passos).
"""

In [ ]:
# salvar melhor pipeline
if best_model_name == 'xgb_tuned':
    final_model = best_xgb
elif best_model_name in models:
    final_model = models[best_model_name]
    final_model.fit(X_train, y_train)
else:
    final_model = search.best_estimator_ if 'search' in globals() else None

if final_model is not None:
    joblib.dump(final_model, 'mvp_olimpiadas_best_model.joblib')
    print("Modelo salvo: mvp_olimpiadas_best_model.joblib")
else:
    print("Nenhum modelo final disponível para salvar.")
